In [ ]:
!pip install transformers==4.51.2 sacremoses

# Imports

In [ ]:
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import os
import torch

# Load data

In [ ]:
from google.colab import drive
if not os.path.exists('/gd'):
    drive.mount('/gd')

In [ ]:
# Load the revised dataset
data_revised = pd.read_csv('/gd/MyDrive/NLLB Data/AWAL evaluation sets - FLORES devtest (REVISED).csv')

data_revised["ID"] = data_revised.index

# remove unnecessary columns
data_revised = data_revised[["ID", "English", "Tamazight (Corrected)"]]
data_revised.columns = ["ID", "eng", "zgh"]

data_revised.describe(include='all')

In [ ]:
# Load the original dataset
eng_data_original = load_dataset("openlanguagedata/flores_plus", "eng_Latn", split="devtest")
zgh_data_original = load_dataset("openlanguagedata/flores_plus", "zgh_Tfng", split="devtest")

data_original = pd.DataFrame({
    "ID": eng_data_original["id"],
    "eng": eng_data_original["text"],
    "zgh": zgh_data_original["text"],
})

data_original.describe(include='all')

In [ ]:
# sanity check
assert data_original['eng'].equals(data_revised['eng']), "English texts do not match!"
assert not data_original['zgh'].equals(data_revised['zgh']), "Revised and original Tamazight texts should not match!"

# Preprocess data

In [ ]:
# replace the 'ⵒ', 'ⵁ', and 'ⴴ' letters with 'ⴱ', 'ⵀ', and 'ⵖ' respectively, since they're not in the model's vocabulary
data_revised['zgh'] = data_revised['zgh'].str.replace('ⵒ', 'ⴱ')
data_revised['zgh'] = data_revised['zgh'].str.replace('ⵁ', 'ⵀ')
data_revised['zgh'] = data_revised['zgh'].str.replace('ⴴ', 'ⵖ')

data_original['zgh'] = data_original['zgh'].str.replace('ⵒ', 'ⴱ')
data_original['zgh'] = data_original['zgh'].str.replace('ⵁ', 'ⵀ')
data_original['zgh'] = data_original['zgh'].str.replace('ⴴ', 'ⵖ')

In [ ]:
# this code is adapted from  the Stopes repo of the NLLB team
# https://github.com/facebookresearch/stopes/blob/main/stopes/pipelines/monolingual/monolingual_line_processor.py#L214

import re
import sys
import typing as tp
import unicodedata
from sacremoses import MosesPunctNormalizer


mpn = MosesPunctNormalizer(lang="en")
mpn.substitutions = [
    (re.compile(r), sub) for r, sub in mpn.substitutions
]


def get_non_printing_char_replacer(replace_by: str = " ") -> tp.Callable[[str], str]:
    non_printable_map = {
        ord(c): replace_by
        for c in (chr(i) for i in range(sys.maxunicode + 1))
        # same as \p{C} in perl
        # see https://www.unicode.org/reports/tr44/#General_Category_Values
        if unicodedata.category(c) in {"C", "Cc", "Cf", "Cs", "Co", "Cn"}
    }

    def replace_non_printing_char(line) -> str:
        return line.translate(non_printable_map)

    return replace_non_printing_char

replace_nonprint = get_non_printing_char_replacer(" ")

def preproc(text):
    clean = mpn.normalize(text)
    clean = replace_nonprint(clean)
    # replace 𝓕𝔯𝔞𝔫𝔠𝔢𝔰𝔠𝔞 by Francesca
    clean = unicodedata.normalize("NFKC", clean)
    return clean

In [ ]:
# apply preprocessing
data_revised['eng'] = data_revised['eng'].apply(preproc)
data_revised['zgh'] = data_revised['zgh'].apply(preproc)

data_original['eng'] = data_original['eng'].apply(preproc)
data_original['zgh'] = data_original['zgh'].apply(preproc)

# Evaluation

In [ ]:
NLLB_LANG_CODES = {
    "zgh": "tzm_Tfng",
    "eng": "eng_Latn"
}

In [ ]:
MAX_LENGTH = 238
NUM_BEAMS = 8

In [ ]:
MODEL_ID = "Tamazight-NLP/NLLB-200-600M-Tamazight-All-Data-1.25-epoch"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, token=True, src_lang="eng_Latn"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, token=True).to(device)

In [ ]:
def get_nllb_translations(data, tokenizer, model, source_lang_code, target_lang_code, **kwargs):
    translations = []

    tokenizer.src_lang = NLLB_LANG_CODES[source_lang_code]

    for _, row in tqdm(data.iterrows(), total=len(data)):
        id = row["ID"]
        text = row[source_lang_code]

        inputs = tokenizer(text, return_tensors="pt").to(model.device)

        translated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.convert_tokens_to_ids(NLLB_LANG_CODES[target_lang_code]),
            **kwargs
            )
        translation = tokenizer.batch_decode(translated_tokens, skip_special_tokens=True)[0]

        translations.append({
            "ID": id,
            "prediction": translation
        })

    return pd.DataFrame(translations)

# Revised data ENG->ZGH

In [ ]:
eng_zgh_translations = get_nllb_translations(data_revised, tokenizer, model, "eng", "zgh", max_length=MAX_LENGTH, num_beams=NUM_BEAMS)
eng_zgh_translations

In [ ]:
# save the answers to a CSV file
eng_zgh_translations.to_csv(f"nllb-600m-all-data-1_25-epoch-{NUM_BEAMS}-beam-eng-zgh-revised.csv", index=False)

# Revised data ZGH->ENG

In [ ]:
zgh_eng_translations = get_nllb_translations(data_revised, tokenizer, model, "zgh", "eng", max_length=MAX_LENGTH, num_beams=NUM_BEAMS)
zgh_eng_translations

In [ ]:
# save the answers to a CSV file
zgh_eng_translations.to_csv(f"nllb-600m-all-data-1_25-epoch-{NUM_BEAMS}-beam-zgh-eng-revised.csv", index=False)

# Original data ZGH->ENG

In [ ]:
original_zgh_eng_translations = get_nllb_translations(data_original, tokenizer, model, "zgh", "eng", max_length=MAX_LENGTH, num_beams=NUM_BEAMS)
original_zgh_eng_translations

In [ ]:
# save the answers to a CSV file
original_zgh_eng_translations.to_csv(f"nllb-600m-all-data-1_25-epoch-{NUM_BEAMS}-beam-zgh-eng-original.csv", index=False)